In [59]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from methyldl.data.pseudo_bulk_generation import generate_pseudo_bulk
from methyldl.deconvolution.uxm import load_atlas
import pickle
import json
import os

In [4]:
soft_labels_data_path = "/home/luna.kuleuven.be/u0169940/Repos/methyldl/Tutorials/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels_pooled_jakkard_no_data_leak_d041/pseudobulk/predicted_reads.pkl"
hard_labels_data_path = "/home/luna.kuleuven.be/u0169940/Repos/methyldl/Tutorials/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_hard_labels_minibatch_balanced/pseudobulk/predicted_reads.pkl"

In [5]:
with open(soft_labels_data_path, "rb") as f:
    soft_train, soft_valid,soft_test = pickle.load(f)

In [6]:
with open(hard_labels_data_path, "rb") as f:
    hard_train, hard_valid,hard_test = pickle.load(f)

In [53]:
atlas_location = "../../UXM_deconv/supplemental/Atlas.U25.l4.hg38.full.tsv"
atlas, ref_cells = load_atlas(atlas_location)

with open("../App/labels_dict.json", "rb") as f:
    labels_dict = json.load(f) 
labels_dict = {int(key):value for (key,value) in labels_dict.items()}
labels_dict_reversed = {y: x for (x, y) in labels_dict.items()}

In [56]:
n_io_examples = 39 # Number of examples per split
n_read_per_split = 4.75*1e5 # Number of reads based on which information is aggregated. 
pure_ios_soft = []
for i in tqdm(range(n_io_examples)):
    proportions = np.zeros(39)
    proportions[i] = 1
    labels = np.where(np.array(proportions)>0)[0]
    proportions = proportions[proportions>0]
    labels, proportions, subs,uxm_data = generate_pseudo_bulk(n_read_per_split, labels,proportions, soft_train,soft_valid, soft_test,atlas,ref_cells,labels_dict_reversed, return_reads = False, num_labels=39)
    pure_ios_soft.append((proportions, subs,uxm_data))

100%|██████████| 39/39 [01:03<00:00,  1.63s/it]


In [57]:
n_io_examples = 39 # Number of examples per split
n_read_per_split = 4.75*1e5 # Number of reads based on which information is aggregated. 
pure_ios_hard = []
for i in tqdm(range(n_io_examples)):
    proportions = np.zeros(39)
    proportions[i] = 1
    labels = np.where(np.array(proportions)>0)[0]
    proportions = proportions[proportions>0]
    labels, proportions, subs,uxm_data = generate_pseudo_bulk(n_read_per_split, labels,proportions, hard_train,hard_valid, hard_test,atlas,ref_cells,labels_dict_reversed, return_reads = False, num_labels=39)
    pure_ios_hard.append((proportions, subs,uxm_data))

100%|██████████| 39/39 [01:04<00:00,  1.66s/it]


In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def compare_model_predictions(data1, data2, true_props_array=None, class_name_key='dmr_ctype', class_idx_key='dmr_ctype_label', save_path=None, model_names = ["Model1", "Model2"]):
    """
    Compares prediction matrices from two models with potentially differing number of classes.
    Aligns both to a master union of classes, padding missing entries with NaNs.
    """
    # 1. Convert to DataFrames
    df1 = pd.DataFrame(data1)
    df2 = pd.DataFrame(data2)
    
    # 2. Build the master list of classes (Union of both datasets)
    all_labels = pd.concat([df1[[class_idx_key, class_name_key]], 
                            df2[[class_idx_key, class_name_key]]]).drop_duplicates()
    all_labels = all_labels.sort_values(class_idx_key).reset_index(drop=True)
    
    master_indices = all_labels[class_idx_key].tolist()
    master_names = all_labels[class_name_key].tolist()
    master_pred_cols = [f'prediction_{idx}_wavg' for idx in master_indices]
    
    # 3. Align DataFrames to the master list
    def align_dataframe(df, master_idx, master_cols):
        # Set index to class labels and reindex to master list (fills missing rows with NaN)
        aligned = df.set_index(class_idx_key).reindex(master_idx)
        # Ensure all master prediction columns exist (fills missing columns with NaN)
        for col in master_cols:
            if col not in aligned.columns:
                aligned[col] = np.nan
        # Reorder columns to match master list strictly
        return aligned[master_cols].values

    mat1 = align_dataframe(df1, master_indices, master_pred_cols)
    mat2 = align_dataframe(df2, master_indices, master_pred_cols)
    
    # Calculate difference (NaNs will propagate, leaving blank spots where classes are missing)
    diff_mat = mat2 - mat1
    
    # 4. Extract diagonals cleanly from the aligned square matrices
    diag1 = np.diag(mat1)
    diag2 = np.diag(mat2)

    # 6. Visualization
    fig = plt.figure(figsize=(20, 14))
    sns.set_theme(style="whitegrid")
    
    # Heatmap kwargs to handle NaNs cleanly
    heatmap_kwargs = {'cmap': 'viridis', 'yticklabels': master_names, 'xticklabels': master_names, 'mask': np.isnan(mat1)}
    
    ax1 = fig.add_subplot(2, 2, 1)
    sns.heatmap(mat1, ax=ax1, **heatmap_kwargs)
    ax1.set_title(f'{model_names[0]}: Full Prediction Matrix', fontsize=14, fontweight='bold')
    ax1.set_ylabel('True Class')
    
    heatmap_kwargs['mask'] = np.isnan(mat2)
    ax2 = fig.add_subplot(2, 2, 2)
    sns.heatmap(mat2, ax=ax2, **heatmap_kwargs)
    ax2.set_title(f'{model_names[1]}: Full Prediction Matrix (Missing Classes Blank)', fontsize=14, fontweight='bold')
    
    ax3 = fig.add_subplot(2, 2, 3)
    vmax = np.nanmax(np.abs(diff_mat)) # Use nanmax to ignore NaNs
    sns.heatmap(diff_mat, cmap='coolwarm', ax=ax3, vmin=-vmax, vmax=vmax, 
                yticklabels=master_names, xticklabels=master_names, mask=np.isnan(diff_mat))
    ax3.set_title('Difference (Model 2 - Model 1)', fontsize=14, fontweight='bold')
    ax3.set_ylabel('True Class')
    
    # Diagonal Plot
    ax4 = fig.add_subplot(2, 2, 4)
    x = np.arange(len(master_names))
    width = 0.35
    
    ax4.bar(x - width/2, diag1, width, label='Model 1', color='#440154', alpha=0.8)
    ax4.bar(x + width/2, diag2, width, label='Model 2', color='#21918c', alpha=0.8)
    
    if true_props_array is not None:
        ax4.plot(x, true_props_array, color='red', marker='D', markersize=8, 
                 linestyle='None', zorder=3, label='Ground Truth Proportion')
                 
    ax4.set_xticks(x)
    ax4.set_xticklabels(master_names, rotation=90)
    ax4.set_title('Diagonal Scores vs. Ground Truth', fontsize=14, fontweight='bold')
    ax4.legend()

    plt.tight_layout()

    
    # 7. Build Analytical DataFrame
    df_dict = {
        'dmr_ctype_label': master_indices,
        'dmr_ctype': master_names,
        'Model_1_Target_Score': diag1,
        'Model_2_Target_Score': diag2,
        'Difference_(M2-M1)': diag2 - diag1
    }
    
    if true_props_array is not None:
        df_dict['True_Proportion'] = true_props_array
        df_dict['Model_1_Error'] = diag1 - true_props_array
        df_dict['Model_2_Error'] = diag2 - true_props_array
        
    diag_comparison_df = pd.DataFrame(df_dict)
    if save_path:
        # Ensure directory exists
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close(fig) # CRITICAL: Releases memory
        return diff_mat, diag_comparison_df
    else:
        plt.show()
        return diff_mat, diag_comparison_df


In [60]:
for i in range(39):
    # Construct a unique filename for each iteration
    file_name = f"comparison_pure_{labels_dict[i]}.png"
    full_path = os.path.join("hard_vs_soft_matrices_new", file_name)
    
    print(f"Processing and saving iteration {i}...")
    
    # Call the modified function
    compare_model_predictions(
        pure_ios_hard[i][1][0], 
        pure_ios_soft[i][1][0], 
        true_props_array=pure_ios_hard[i][0], # Assuming this is your dict/array
        save_path=full_path,
        model_names=["Hard Labeled with Rej. Microbatch balanced (0.5)", "Soft Labeled with Jaccard Pooling (d<=0.41)"]
    )

Processing and saving iteration 0...
Processing and saving iteration 1...
Processing and saving iteration 2...
Processing and saving iteration 3...
Processing and saving iteration 4...
Processing and saving iteration 5...
Processing and saving iteration 6...
Processing and saving iteration 7...
Processing and saving iteration 8...
Processing and saving iteration 9...
Processing and saving iteration 10...
Processing and saving iteration 11...
Processing and saving iteration 12...
Processing and saving iteration 13...
Processing and saving iteration 14...
Processing and saving iteration 15...
Processing and saving iteration 16...
Processing and saving iteration 17...
Processing and saving iteration 18...
Processing and saving iteration 19...
Processing and saving iteration 20...
Processing and saving iteration 21...
Processing and saving iteration 22...
Processing and saving iteration 23...
Processing and saving iteration 24...
Processing and saving iteration 25...
Processing and saving 